# Shared preprocessing pipeline — Flower image classification

Pipeline usa gli split ufficiali in formato CSV:

- `data/splits/train_folds.csv` con colonne `image_path`, `label`, `fold`;
- `data/splits/test.csv` con colonne `image_path`, `label`;
- `fold == val_fold` diventa validation set;
- `fold != val_fold` diventa training set;
- `directory_debug` è stato rimosso.

La pipeline resta minimale: load image, decode RGB, resize, cast a `float32`, batch, shuffle solo sul training set e prefetch. `Rescaling`, `preprocess_input` e data augmentation model-specific restano nei notebook di modelling.

## 1. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab: skipping Drive mount.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Definizione dei path del progetto

In [ ]:
from pathlib import Path

PROJECT_DIR = "/content/drive/MyDrive/FDL Project"
DATA_DIR = f"{PROJECT_DIR}/data/flower_photos"
SPLIT_DIR = f"{PROJECT_DIR}/data/splits"

TRAIN_FOLDS_CSV = "train_folds.csv"
TEST_CSV = "test.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("SPLIT_DIR:", SPLIT_DIR)
print("TRAIN_FOLDS_CSV:", f"{SPLIT_DIR}/{TRAIN_FOLDS_CSV}")
print("TEST_CSV:", f"{SPLIT_DIR}/{TEST_CSV}")

PROJECT_DIR: /content/drive/MyDrive/FDL Project
DATA_DIR: /content/drive/MyDrive/FDL Project/data/flower_photos
SPLIT_DIR: /content/drive/MyDrive/FDL Project/data/splits
TRAIN_FOLDS_CSV: /content/drive/MyDrive/FDL Project/data/splits/train_folds.csv
TEST_CSV: /content/drive/MyDrive/FDL Project/data/splits/test.csv


## 3. Import librerie

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List, Optional, Tuple, Union

import pandas as pd
import tensorflow as tf
from tensorflow import keras

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## 4. Funzioni della pipeline CSV/fold-based

In [ ]:
"""
Shared preprocessing pipeline for the flower image classification project.

CSV-based logic:
- train_folds.csv contains image_path, label, fold;
- one fold is selected as validation;
- all remaining folds are used for training;
- test.csv contains at least image_path and label;
- directory_debug logic has been removed.

The pipeline keeps preprocessing minimal and shared:
- read image files from image_path;
- decode RGB with 3 channels;
- resize;
- cast to float32;
- optional rescaling to [0, 1];
- shuffle only the training dataset;
- batch and prefetch;
- provide load_common_datasets() as a one-call wrapper for training notebooks.

Model-specific preprocessing, advanced normalization and data augmentation
should stay inside the modelling notebooks/models.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List, Optional, Tuple, Union

import pandas as pd
import tensorflow as tf


AUTOTUNE = tf.data.AUTOTUNE
TRAIN_FOLDS_REQUIRED_COLUMNS = {"image_path", "label", "fold"}
TEST_REQUIRED_COLUMNS = {"image_path", "label"}
VALID_NORMALIZATIONS = {None, "rescale_0_1"}
PATH_COLUMN_ALIASES = ("image_path", "file_path", "filepath", "path")


@dataclass
class PreprocessingConfig:
    """Configuration for the shared CSV/fold-based preprocessing pipeline."""

    project_dir: str = "/content/drive/MyDrive/FDL Project"
    split_dir: Optional[str] = None

    train_folds_csv: str = "train_folds.csv"
    test_csv: str = "test.csv"
    val_fold: int = 0

    img_size: int = 224
    batch_size: int = 32
    seed: int = 42

    normalization: Optional[str] = None
    shuffle_train: bool = True
    check_files: bool = True

    def __post_init__(self) -> None:
        """Fill default directories and validate basic config values."""
        if self.split_dir is None:
            self.split_dir = str(Path(self.project_dir) / "data" / "splits")

        if self.normalization not in VALID_NORMALIZATIONS:
            raise ValueError(
                f"normalization must be one of {VALID_NORMALIZATIONS}, "
                f"got: {self.normalization}"
            )

        if self.img_size <= 0:
            raise ValueError("img_size must be a positive integer.")

        if self.batch_size <= 0:
            raise ValueError("batch_size must be a positive integer.")

        try:
            self.val_fold = int(self.val_fold)
        except (TypeError, ValueError) as exc:
            raise ValueError("val_fold must be an integer.") from exc


def resolve_path(path: Union[str, Path], base_dir: Optional[Union[str, Path]] = None) -> Path:
    """Resolve absolute paths; relative paths are resolved from base_dir."""
    path = Path(path)

    if path.is_absolute():
        return path

    if base_dir is None:
        return path

    return Path(base_dir) / path


def _standardize_path_column(df: pd.DataFrame, csv_path: Union[str, Path]) -> pd.DataFrame:
    """
    Ensure the DataFrame has an image_path column.

    The official expected column is image_path. The aliases are accepted only to
    make the function robust if an older split used file_path/path/filepath.
    """
    df = df.copy()

    if "image_path" in df.columns:
        return df

    for alias in PATH_COLUMN_ALIASES:
        if alias in df.columns:
            return df.rename(columns={alias: "image_path"})

    raise ValueError(
        f"{Path(csv_path).name} must contain an image_path column. "
        f"Accepted aliases: {PATH_COLUMN_ALIASES}. Found columns: {list(df.columns)}"
    )


def load_split_csv(
    csv_path: Union[str, Path],
    required_columns: Iterable[str],
    split_name: str,
) -> pd.DataFrame:
    """Load and validate a split CSV."""
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"{split_name} CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    df = _standardize_path_column(df, csv_path=csv_path)

    required_columns = set(required_columns)
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"{csv_path.name} is missing required columns: {sorted(missing_columns)}"
        )

    if df.empty:
        raise ValueError(f"{csv_path.name} is empty.")

    if df[list(required_columns)].isna().any().any():
        raise ValueError(f"{csv_path.name} contains missing values in required columns.")

    try:
        df["label"] = df["label"].astype(int)
    except ValueError as exc:
        raise ValueError(f"{csv_path.name}: label column must contain integers.") from exc

    if (df["label"] < 0).any():
        raise ValueError(f"{csv_path.name}: labels must be non-negative integers.")

    if "fold" in required_columns:
        try:
            df["fold"] = df["fold"].astype(int)
        except ValueError as exc:
            raise ValueError(f"{csv_path.name}: fold column must contain integers.") from exc

    df["image_path"] = df["image_path"].astype(str)

    # Defensive check: each row must represent ONE image.
    # If a CSV cell contains a Python-like list such as
    # "['tulips/img1.jpg', 'tulips/img2.jpg']", the split CSV was generated incorrectly.
    list_like_mask = df["image_path"].str.contains(r"\[.*\]", regex=True, na=False)
    if list_like_mask.any():
        bad_example = df.loc[list_like_mask, "image_path"].iloc[0]
        raise ValueError(
            f"{csv_path.name} contains list-like values inside image_path. "
            "Expected one row per image, not a list of images in one cell. "
            f"Example bad value: {bad_example[:300]}..."
        )
    return df


def infer_class_names(*dfs: pd.DataFrame) -> List[str]:
    """
    Infer class names from the CSVs.

    If class_name exists, it is used when each numeric label maps to one unique
    name. Otherwise labels are represented as class_0, class_1, ... .
    """
    non_empty_dfs = [df for df in dfs if df is not None and not df.empty]
    if not non_empty_dfs:
        raise ValueError("Cannot infer classes from empty DataFrames.")

    all_df = pd.concat(non_empty_dfs, ignore_index=True)

    if "class_name" in all_df.columns:
        mapping = (
            all_df[["label", "class_name"]]
            .dropna()
            .drop_duplicates()
            .sort_values("label")
        )

        duplicated_labels = mapping["label"].duplicated(keep=False)
        if not duplicated_labels.any() and not mapping.empty:
            max_label = int(all_df["label"].max())
            label_to_name = dict(zip(mapping["label"].astype(int), mapping["class_name"]))
            return [str(label_to_name.get(i, f"class_{i}")) for i in range(max_label + 1)]

    max_label = int(all_df["label"].max())
    return [f"class_{i}" for i in range(max_label + 1)]


def split_train_val_from_folds(
    train_folds_df: pd.DataFrame,
    val_fold: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Use one fold as validation and all other folds as training."""
    available_folds = sorted(train_folds_df["fold"].unique().tolist())

    if val_fold not in available_folds:
        raise ValueError(
            f"val_fold={val_fold} is not present in train_folds.csv. "
            f"Available folds: {available_folds}"
        )

    train_df = train_folds_df[train_folds_df["fold"] != val_fold].copy()
    val_df = train_folds_df[train_folds_df["fold"] == val_fold].copy()

    if train_df.empty:
        raise ValueError("Training split is empty. Choose a different val_fold.")

    if val_df.empty:
        raise ValueError("Validation split is empty. Choose a different val_fold.")

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def add_absolute_paths(
    df: pd.DataFrame,
    project_dir: Optional[Union[str, Path]] = None,
    check_files: bool = True,
) -> pd.DataFrame:
    """Add an absolute_path column resolved from project_dir."""
    if "image_path" not in df.columns:
        raise ValueError("DataFrame must contain an image_path column.")

    df = df.copy()
    base_dir = Path(project_dir) if project_dir is not None else None

    absolute_paths = [
        str(resolve_path(path, base_dir=base_dir))
        for path in df["image_path"].astype(str).tolist()
    ]

    if check_files:
        missing_paths = [path for path in absolute_paths if not Path(path).exists()]
        if missing_paths:
            preview = "\n".join(missing_paths[:10])
            raise FileNotFoundError(
                f"{len(missing_paths)} image files were not found. "
                f"First missing paths:\n{preview}\n\n"
                "Check whether image_path is absolute or relative to PROJECT_DIR."
            )

    df["absolute_path"] = absolute_paths
    return df


def load_dataframes(
    config: PreprocessingConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Load train_folds.csv and test.csv, then create train/val/test DataFrames.

    Returns:
        train_df, val_df, test_df, class_names
    """
    split_dir = Path(config.split_dir)

    train_folds_path = resolve_path(config.train_folds_csv, base_dir=split_dir)
    test_path = resolve_path(config.test_csv, base_dir=split_dir)

    train_folds_df = load_split_csv(
        train_folds_path,
        required_columns=TRAIN_FOLDS_REQUIRED_COLUMNS,
        split_name="train_folds",
    )
    test_df = load_split_csv(
        test_path,
        required_columns=TEST_REQUIRED_COLUMNS,
        split_name="test",
    )

    class_names = infer_class_names(train_folds_df, test_df)
    train_df, val_df = split_train_val_from_folds(
        train_folds_df=train_folds_df,
        val_fold=config.val_fold,
    )

    train_df = add_absolute_paths(
        train_df,
        project_dir=config.project_dir,
        check_files=config.check_files,
    )
    val_df = add_absolute_paths(
        val_df,
        project_dir=config.project_dir,
        check_files=config.check_files,
    )
    test_df = add_absolute_paths(
        test_df,
        project_dir=config.project_dir,
        check_files=config.check_files,
    )

    return train_df, val_df, test_df, class_names


def decode_image(image_path: tf.Tensor) -> tf.Tensor:
    """Read an image file and decode it as RGB with 3 channels."""
    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False,
    )
    image.set_shape([None, None, 3])
    return image


def preprocess_image(
    image: tf.Tensor,
    img_size: int = 224,
    normalization: Optional[str] = None,
) -> tf.Tensor:
    """Resize image and cast to float32; normalization is optional."""
    image = tf.image.resize(image, [img_size, img_size], method="bilinear")
    image = tf.cast(image, tf.float32)

    if normalization == "rescale_0_1":
        image = image / 255.0
    elif normalization is None:
        pass
    else:
        raise ValueError(
            "normalization must be None or 'rescale_0_1'. "
            f"Got: {normalization}"
        )

    image.set_shape([img_size, img_size, 3])
    return image


def _load_and_preprocess_example(
    image_path: tf.Tensor,
    label: tf.Tensor,
    img_size: int,
    normalization: Optional[str],
) -> Tuple[tf.Tensor, tf.Tensor]:
    """Load one example and return image tensor plus label."""
    image = decode_image(image_path)
    image = preprocess_image(image, img_size=img_size, normalization=normalization)
    label = tf.cast(label, tf.int32)
    return image, label


def build_dataset_from_dataframe(
    df: pd.DataFrame,
    config: PreprocessingConfig,
    training: bool = False,
) -> tf.data.Dataset:
    """Build a tf.data.Dataset from a DataFrame with file paths and labels."""
    path_column = "absolute_path" if "absolute_path" in df.columns else "image_path"

    image_paths = df[path_column].astype(str).tolist()
    labels = df["label"].astype(int).tolist()

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if training and config.shuffle_train:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=config.seed,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        lambda image_path, label: _load_and_preprocess_example(
            image_path=image_path,
            label=label,
            img_size=config.img_size,
            normalization=config.normalization,
        ),
        num_parallel_calls=AUTOTUNE,
    )

    dataset = dataset.batch(config.batch_size)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


def build_datasets(
    config: PreprocessingConfig,
) -> Tuple[tf.data.Dataset, tf.data.Dataset, tf.data.Dataset, List[str]]:
    """
    Build train_ds, val_ds and test_ds from CSV splits.

    train_ds uses all rows where fold != config.val_fold.
    val_ds uses all rows where fold == config.val_fold.
    test_ds uses all rows from test.csv.
    """
    train_df, val_df, test_df, class_names = load_dataframes(config)

    train_ds = build_dataset_from_dataframe(train_df, config=config, training=True)
    val_ds = build_dataset_from_dataframe(val_df, config=config, training=False)
    test_ds = build_dataset_from_dataframe(test_df, config=config, training=False)

    return train_ds, val_ds, test_ds, class_names


def make_common_config(
    project_dir: Union[str, Path] = "/content/drive/MyDrive/FDL Project",
    val_fold: int = 0,
    img_size: int = 224,
    batch_size: int = 32,
    normalization: Optional[str] = None,
    shuffle_train: bool = True,
    check_files: bool = True,
    seed: int = 42,
) -> PreprocessingConfig:
    """
    Create the standard project preprocessing configuration.

    This wrapper fixes the common project split paths:
    - data/splits/train_folds.csv
    - data/splits/test.csv

    Use this in modelling notebooks when you want the shared setup without
    rewriting SPLIT_DIR, TRAIN_FOLDS_CSV, TEST_CSV and PreprocessingConfig.
    """
    project_dir = Path(project_dir)
    split_dir = project_dir / "data" / "splits"

    return PreprocessingConfig(
        project_dir=str(project_dir),
        split_dir=str(split_dir),
        train_folds_csv="train_folds.csv",
        test_csv="test.csv",
        val_fold=val_fold,
        img_size=img_size,
        batch_size=batch_size,
        seed=seed,
        normalization=normalization,
        shuffle_train=shuffle_train,
        check_files=check_files,
    )


def load_common_datasets(
    project_dir: Union[str, Path] = "/content/drive/MyDrive/FDL Project",
    val_fold: int = 0,
    img_size: int = 224,
    batch_size: int = 32,
    normalization: Optional[str] = None,
    shuffle_train: bool = True,
    check_files: bool = True,
    seed: int = 42,
) -> Tuple[tf.data.Dataset, tf.data.Dataset, tf.data.Dataset, List[str]]:
    """
    Build the shared train, validation and test datasets with one function call.

    Returns:
        train_ds, val_ds, test_ds, class_names

    Typical use in a modelling notebook:
        train_ds, val_ds, test_ds, class_names = load_common_datasets(
            project_dir="/content/drive/MyDrive/FDL Project",
            val_fold=0,
            img_size=224,
            batch_size=32,
            normalization=None,
        )
    """
    config = make_common_config(
        project_dir=project_dir,
        val_fold=val_fold,
        img_size=img_size,
        batch_size=batch_size,
        normalization=normalization,
        shuffle_train=shuffle_train,
        check_files=check_files,
        seed=seed,
    )
    return build_datasets(config)


def summarize_dataset(
    dataset: tf.data.Dataset,
    class_names: Optional[List[str]] = None,
    max_batches: int = 1,
) -> None:
    """Print basic information about one or more dataset batches."""
    print("Dataset summary")
    print("-" * 60)

    if class_names is not None:
        print(f"Number of classes: {len(class_names)}")
        print(f"Class names: {class_names}")

    for batch_idx, (images, labels) in enumerate(dataset.take(max_batches), start=1):
        print(f"\nBatch {batch_idx}")
        print(f"images.shape: {images.shape}")
        print(f"images.dtype: {images.dtype}")
        print(f"labels.shape: {labels.shape}")
        print(f"labels.dtype: {labels.dtype}")
        print(f"image min: {tf.reduce_min(images).numpy():.3f}")
        print(f"image max: {tf.reduce_max(images).numpy():.3f}")
        print(f"first labels: {labels[:10].numpy()}")

    print("-" * 60)


def summarize_split_dataframes(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    class_names: Optional[List[str]] = None,
) -> pd.DataFrame:
    """Return a compact class distribution table for train, val and test."""
    n_classes = len(class_names) if class_names is not None else int(
        max(train_df["label"].max(), val_df["label"].max(), test_df["label"].max()) + 1
    )

    rows = []
    for label in range(n_classes):
        class_name = class_names[label] if class_names is not None else f"class_{label}"
        rows.append(
            {
                "label": label,
                "class_name": class_name,
                "train": int((train_df["label"] == label).sum()),
                "val": int((val_df["label"] == label).sum()),
                "test": int((test_df["label"] == label).sum()),
            }
        )

    summary = pd.DataFrame(rows)
    total_row = pd.DataFrame(
        [
            {
                "label": "total",
                "class_name": "total",
                "train": len(train_df),
                "val": len(val_df),
                "test": len(test_df),
            }
        ]
    )
    return pd.concat([summary, total_row], ignore_index=True)


## 5. Configurazione finale da `train_folds.csv` e `test.csv`

Questa configurazione esplicita è utile nel notebook di preprocessing per testare e mostrare tutti i parametri.

Nei notebook di training degli altri, invece, potranno usare direttamente la funzione wrapper `load_common_datasets(...)`, senza riscrivere `SPLIT_DIR`, `TRAIN_FOLDS_CSV`, `TEST_CSV` e `PreprocessingConfig`.

----------------------

This explicit configuration is useful in the preprocessing notebook to test and display all the parameters.

In the other training notebooks, however, they will be able to directly use the wrapper function `load_common_datasets(...)`, without rewriting `SPLIT_DIR`, `TRAIN_FOLDS_CSV`, `TEST_CSV`, and `PreprocessingConfig`.


In [ ]:
final_config = PreprocessingConfig(
    project_dir=PROJECT_DIR,
    split_dir=SPLIT_DIR,
    train_folds_csv=TRAIN_FOLDS_CSV,
    test_csv=TEST_CSV,
    val_fold=0,
    img_size=224,
    batch_size=32,
    normalization=None,   # default: pixel float32 in range circa [0, 255]
    shuffle_train=True,
    check_files=True,
)

print(final_config)

PreprocessingConfig(project_dir='/content/drive/MyDrive/FDL Project', split_dir='/content/drive/MyDrive/FDL Project/data/splits', train_folds_csv='train_folds.csv', test_csv='test.csv', val_fold=0, img_size=224, batch_size=32, seed=42, normalization=None, shuffle_train=True, check_files=True)


## 6. Controllo veloce dei CSV / Quick CSV Check

In [ ]:
train_folds_path = Path(SPLIT_DIR) / TRAIN_FOLDS_CSV
test_path = Path(SPLIT_DIR) / TEST_CSV

train_folds_preview = pd.read_csv(train_folds_path)
test_preview = pd.read_csv(test_path)

print("train_folds.csv shape:", train_folds_preview.shape)
print("test.csv shape:", test_preview.shape)
print("\ntrain_folds.csv columns:", list(train_folds_preview.columns))
print("test.csv columns:", list(test_preview.columns))
print("\nAvailable folds:", sorted(train_folds_preview["fold"].unique().tolist()))

display(train_folds_preview.head())
display(test_preview.head())

train_folds.csv shape: (2932, 3)
test.csv shape: (733, 2)

train_folds.csv columns: ['image_path', 'label', 'fold']
test.csv columns: ['image_path', 'label']

Available folds: [0, 1, 2, 3, 4]


,image_path,label,fold
0,/content/drive/MyDrive/FDL Project/data/flower...,1,0
1,/content/drive/MyDrive/FDL Project/data/flower...,4,1
2,/content/drive/MyDrive/FDL Project/data/flower...,2,2
3,/content/drive/MyDrive/FDL Project/data/flower...,2,4
4,/content/drive/MyDrive/FDL Project/data/flower...,1,2


,image_path,label
0,/content/drive/MyDrive/FDL Project/data/flower...,1
1,/content/drive/MyDrive/FDL Project/data/flower...,1
2,/content/drive/MyDrive/FDL Project/data/flower...,2
3,/content/drive/MyDrive/FDL Project/data/flower...,0
4,/content/drive/MyDrive/FDL Project/data/flower...,0


## 7. Creazione dei DataFrame train / validation / test | Creating Train Val and Test Dataframe

Qui avviene lo split vero: `fold == val_fold` diventa validation, tutto il resto diventa train.

This is where the actual split happens: `fold == val_fold` becomes the validation set, and everything else becomes the training set.

In [ ]:
train_df, val_df, test_df, class_names = load_dataframes(final_config)

print("Class names inferred from labels:", class_names)
print("Number of classes:", len(class_names))
print("\nRows:")
print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

split_summary = summarize_split_dataframes(train_df, val_df, test_df, class_names)
display(split_summary)

Class names inferred from labels: ['class_0', 'class_1', 'class_2', 'class_3', 'class_4']
Number of classes: 5

Rows:
train: 2345
val: 587
test: 733


,label,class_name,train,val,test
0,0,class_0,511,127,160
1,1,class_1,574,144,180
2,2,class_2,404,101,126
3,3,class_3,446,112,139
4,4,class_4,410,103,128
5,total,total,2345,587,733


## 8. Creazione dei `tf.data.Dataset`

In [ ]:
train_ds, val_ds, test_ds, class_names = build_datasets(final_config)

print("Train dataset:")
summarize_dataset(train_ds, class_names=class_names, max_batches=1)

print("\nValidation dataset:")
summarize_dataset(val_ds, class_names=class_names, max_batches=1)

print("\nTest dataset:")
summarize_dataset(test_ds, class_names=class_names, max_batches=1)

Train dataset:
Dataset summary
------------------------------------------------------------
Number of classes: 5
Class names: ['class_0', 'class_1', 'class_2', 'class_3', 'class_4']

Batch 1
images.shape: (32, 224, 224, 3)
images.dtype: <dtype: 'float32'>
labels.shape: (32,)
labels.dtype: <dtype: 'int32'>
image min: 0.000
image max: 255.000
first labels: [0 3 3 0 0 3 1 0 1 3]
------------------------------------------------------------

Validation dataset:
Dataset summary
------------------------------------------------------------
Number of classes: 5
Class names: ['class_0', 'class_1', 'class_2', 'class_3', 'class_4']

Batch 1
images.shape: (32, 224, 224, 3)
images.dtype: <dtype: 'float32'>
labels.shape: (32,)
labels.dtype: <dtype: 'int32'>
image min: 0.000
image max: 255.000
first labels: [1 1 4 2 1 0 2 4 4 1]
------------------------------------------------------------

Test dataset:
Dataset summary
------------------------------------------------------------
Number of classes: 5
C

## 9. Smoke test finale

In [ ]:
images, labels = next(iter(train_ds))

print("Expected image batch shape with default config: (batch_size, 224, 224, 3)")
print("images.shape:", images.shape)
print("labels.shape:", labels.shape)
print("images.dtype:", images.dtype)
print("labels.dtype:", labels.dtype)

assert images.shape[-1] == 3
assert images.dtype == tf.float32
assert labels.dtype == tf.int32

print("Smoke test passed.")

Expected image batch shape with default config: (batch_size, 224, 224, 3)
images.shape: (32, 224, 224, 3)
labels.shape: (32,)
images.dtype: <dtype: 'float32'>
labels.dtype: <dtype: 'int32'>
Smoke test passed.


## 10. Esempio di uso per baseline CNN

La pipeline lascia i pixel in `float32` circa `[0, 255]`. Per una baseline CNN from scratch puoi mettere `Rescaling(1./255)` dentro il modello.

In [ ]:
num_classes = len(class_names)

inputs = keras.Input(shape=(224, 224, 3))
x = keras.layers.Rescaling(1.0 / 255)(inputs)

x = keras.layers.Conv2D(32, 3, activation="relu", padding="same")(x)
x = keras.layers.MaxPooling2D()(x)
x = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = keras.layers.MaxPooling2D()(x)
x = keras.layers.GlobalAveragePooling2D()(x)

outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

baseline_model = keras.Model(inputs, outputs, name="baseline_cnn")

baseline_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

baseline_model.summary()

# Esempio:
# baseline_model.fit(train_ds, validation_data=val_ds, epochs=10)

Model: "baseline_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,717 (77.02 KB)

 Trainable params: 19,717 (77.02 KB)

 Non-trainable params: 0 (0.00 B)

## 11. Esempio di uso per transfer learning

Per un pretrained model, il preprocessing model-specific rimane nel modello. Esempio con `ResNet50`: `keras.applications.resnet50.preprocess_input`.

In [ ]:
num_classes = len(class_names)

inputs = keras.Input(shape=(224, 224, 3))

# ResNet50 preprocess_input si aspetta input float32 in range circa [0, 255].
x = keras.applications.resnet50.preprocess_input(inputs)

backbone = keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3),
)

backbone.trainable = False

x = backbone(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

transfer_model = keras.Model(inputs, outputs, name="resnet50_transfer")

transfer_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

transfer_model.summary()

# Esempio:
# transfer_model.fit(train_ds, validation_data=val_ds, epochs=10)

Model: "resnet50_transfer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 224, 224)  │          0 │ input_layer_4[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 224, 224)  │          0 │ input_layer_4[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 224, 224)  │          0 │ input_layer_4[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add_1[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 5)         │     10,245 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

## 12. Normalization opzionale

Default consigliato:

```python
normalization=None
```

Per test rapidi o baseline semplice puoi usare:

```python
normalization="rescale_0_1"
```

Attenzione: se usi `normalization="rescale_0_1"` nella pipeline, non aggiungere di nuovo `Rescaling(1./255)` nel modello.

In [ ]:
rescaled_config = PreprocessingConfig(
    project_dir=PROJECT_DIR,
    split_dir=SPLIT_DIR,
    train_folds_csv=TRAIN_FOLDS_CSV,
    test_csv=TEST_CSV,
    val_fold=0,
    img_size=224,
    batch_size=32,
    normalization="rescale_0_1",
    shuffle_train=True,
    check_files=True,
)

rescaled_train_ds, rescaled_val_ds, rescaled_test_ds, rescaled_class_names = build_datasets(rescaled_config)
summarize_dataset(rescaled_train_ds, class_names=rescaled_class_names, max_batches=1)

Dataset summary
------------------------------------------------------------
Number of classes: 5
Class names: ['class_0', 'class_1', 'class_2', 'class_3', 'class_4']

Batch 1
images.shape: (32, 224, 224, 3)
images.dtype: <dtype: 'float32'>
labels.shape: (32,)
labels.dtype: <dtype: 'int32'>
image min: 0.000
image max: 1.000
first labels: [0 3 3 0 0 3 1 0 1 3]
------------------------------------------------------------


## 13. Export in `scripts/preprocessing.py`

Questa cella salva le funzioni principali in:

```text
FDL Project/scripts/preprocessing.py
```

Così gli altri notebook possono fare:

```python
import sys
sys.path.append(f"{PROJECT_DIR}/scripts")
from preprocessing import PreprocessingConfig, build_datasets
```

In [ ]:
scripts_dir = Path(PROJECT_DIR) / "scripts"
scripts_dir.mkdir(parents=True, exist_ok=True)

preprocessing_py_path = scripts_dir / "preprocessing.py"

preprocessing_py_code = '"""\nShared preprocessing pipeline for the flower image classification project.\n\nCSV-based logic:\n- train_folds.csv contains image_path, label, fold;\n- one fold is selected as validation;\n- all remaining folds are used for training;\n- test.csv contains at least image_path and label;\n- directory_debug logic has been removed.\n\nThe pipeline keeps preprocessing minimal and shared:\n- read image files from image_path;\n- decode RGB with 3 channels;\n- resize;\n- cast to float32;\n- optional rescaling to [0, 1];\n- shuffle only the training dataset;\n- batch and prefetch;\n- provide load_common_datasets() as a one-call wrapper for training notebooks.\n\nModel-specific preprocessing, advanced normalization and data augmentation\nshould stay inside the modelling notebooks/models.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable, List, Optional, Tuple, Union\n\nimport pandas as pd\nimport tensorflow as tf\n\n\nAUTOTUNE = tf.data.AUTOTUNE\nTRAIN_FOLDS_REQUIRED_COLUMNS = {"image_path", "label", "fold"}\nTEST_REQUIRED_COLUMNS = {"image_path", "label"}\nVALID_NORMALIZATIONS = {None, "rescale_0_1"}\nPATH_COLUMN_ALIASES = ("image_path", "file_path", "filepath", "path")\n\n\n@dataclass\nclass PreprocessingConfig:\n    """Configuration for the shared CSV/fold-based preprocessing pipeline."""\n\n    project_dir: str = "/content/drive/MyDrive/FDL Project"\n    split_dir: Optional[str] = None\n\n    train_folds_csv: str = "train_folds.csv"\n    test_csv: str = "test.csv"\n    val_fold: int = 0\n\n    img_size: int = 224\n    batch_size: int = 32\n    seed: int = 42\n\n    normalization: Optional[str] = None\n    shuffle_train: bool = True\n    check_files: bool = True\n\n    def __post_init__(self) -> None:\n        """Fill default directories and validate basic config values."""\n        if self.split_dir is None:\n            self.split_dir = str(Path(self.project_dir) / "data" / "splits")\n\n        if self.normalization not in VALID_NORMALIZATIONS:\n            raise ValueError(\n                f"normalization must be one of {VALID_NORMALIZATIONS}, "\n                f"got: {self.normalization}"\n            )\n\n        if self.img_size <= 0:\n            raise ValueError("img_size must be a positive integer.")\n\n        if self.batch_size <= 0:\n            raise ValueError("batch_size must be a positive integer.")\n\n        try:\n            self.val_fold = int(self.val_fold)\n        except (TypeError, ValueError) as exc:\n            raise ValueError("val_fold must be an integer.") from exc\n\n\ndef resolve_path(path: Union[str, Path], base_dir: Optional[Union[str, Path]] = None) -> Path:\n    """Resolve absolute paths; relative paths are resolved from base_dir."""\n    path = Path(path)\n\n    if path.is_absolute():\n        return path\n\n    if base_dir is None:\n        return path\n\n    return Path(base_dir) / path\n\n\ndef _standardize_path_column(df: pd.DataFrame, csv_path: Union[str, Path]) -> pd.DataFrame:\n    """\n    Ensure the DataFrame has an image_path column.\n\n    The official expected column is image_path. The aliases are accepted only to\n    make the function robust if an older split used file_path/path/filepath.\n    """\n    df = df.copy()\n\n    if "image_path" in df.columns:\n        return df\n\n    for alias in PATH_COLUMN_ALIASES:\n        if alias in df.columns:\n            return df.rename(columns={alias: "image_path"})\n\n    raise ValueError(\n        f"{Path(csv_path).name} must contain an image_path column. "\n        f"Accepted aliases: {PATH_COLUMN_ALIASES}. Found columns: {list(df.columns)}"\n    )\n\n\ndef load_split_csv(\n    csv_path: Union[str, Path],\n    required_columns: Iterable[str],\n    split_name: str,\n) -> pd.DataFrame:\n    """Load and validate a split CSV."""\n    csv_path = Path(csv_path)\n\n    if not csv_path.exists():\n        raise FileNotFoundError(f"{split_name} CSV not found: {csv_path}")\n\n    df = pd.read_csv(csv_path)\n    df = _standardize_path_column(df, csv_path=csv_path)\n\n    required_columns = set(required_columns)\n    missing_columns = required_columns - set(df.columns)\n    if missing_columns:\n        raise ValueError(\n            f"{csv_path.name} is missing required columns: {sorted(missing_columns)}"\n        )\n\n    if df.empty:\n        raise ValueError(f"{csv_path.name} is empty.")\n\n    if df[list(required_columns)].isna().any().any():\n        raise ValueError(f"{csv_path.name} contains missing values in required columns.")\n\n    try:\n        df["label"] = df["label"].astype(int)\n    except ValueError as exc:\n        raise ValueError(f"{csv_path.name}: label column must contain integers.") from exc\n\n    if (df["label"] < 0).any():\n        raise ValueError(f"{csv_path.name}: labels must be non-negative integers.")\n\n    if "fold" in required_columns:\n        try:\n            df["fold"] = df["fold"].astype(int)\n        except ValueError as exc:\n            raise ValueError(f"{csv_path.name}: fold column must contain integers.") from exc\n\n    df["image_path"] = df["image_path"].astype(str)\n\n    # Defensive check: each row must represent ONE image.\n    # If a CSV cell contains a Python-like list such as\n    # "[\'tulips/img1.jpg\', \'tulips/img2.jpg\']", the split CSV was generated incorrectly.\n    list_like_mask = df["image_path"].str.contains(r"\\[.*\\]", regex=True, na=False)\n    if list_like_mask.any():\n        bad_example = df.loc[list_like_mask, "image_path"].iloc[0]\n        raise ValueError(\n            f"{csv_path.name} contains list-like values inside image_path. "\n            "Expected one row per image, not a list of images in one cell. "\n            f"Example bad value: {bad_example[:300]}..."\n        )\n    return df\n\n\ndef infer_class_names(*dfs: pd.DataFrame) -> List[str]:\n    """\n    Infer class names from the CSVs.\n\n    If class_name exists, it is used when each numeric label maps to one unique\n    name. Otherwise labels are represented as class_0, class_1, ... .\n    """\n    non_empty_dfs = [df for df in dfs if df is not None and not df.empty]\n    if not non_empty_dfs:\n        raise ValueError("Cannot infer classes from empty DataFrames.")\n\n    all_df = pd.concat(non_empty_dfs, ignore_index=True)\n\n    if "class_name" in all_df.columns:\n        mapping = (\n            all_df[["label", "class_name"]]\n            .dropna()\n            .drop_duplicates()\n            .sort_values("label")\n        )\n\n        duplicated_labels = mapping["label"].duplicated(keep=False)\n        if not duplicated_labels.any() and not mapping.empty:\n            max_label = int(all_df["label"].max())\n            label_to_name = dict(zip(mapping["label"].astype(int), mapping["class_name"]))\n            return [str(label_to_name.get(i, f"class_{i}")) for i in range(max_label + 1)]\n\n    max_label = int(all_df["label"].max())\n    return [f"class_{i}" for i in range(max_label + 1)]\n\n\ndef split_train_val_from_folds(\n    train_folds_df: pd.DataFrame,\n    val_fold: int,\n) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    """Use one fold as validation and all other folds as training."""\n    available_folds = sorted(train_folds_df["fold"].unique().tolist())\n\n    if val_fold not in available_folds:\n        raise ValueError(\n            f"val_fold={val_fold} is not present in train_folds.csv. "\n            f"Available folds: {available_folds}"\n        )\n\n    train_df = train_folds_df[train_folds_df["fold"] != val_fold].copy()\n    val_df = train_folds_df[train_folds_df["fold"] == val_fold].copy()\n\n    if train_df.empty:\n        raise ValueError("Training split is empty. Choose a different val_fold.")\n\n    if val_df.empty:\n        raise ValueError("Validation split is empty. Choose a different val_fold.")\n\n    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)\n\n\ndef add_absolute_paths(\n    df: pd.DataFrame,\n    project_dir: Optional[Union[str, Path]] = None,\n    check_files: bool = True,\n) -> pd.DataFrame:\n    """Add an absolute_path column resolved from project_dir."""\n    if "image_path" not in df.columns:\n        raise ValueError("DataFrame must contain an image_path column.")\n\n    df = df.copy()\n    base_dir = Path(project_dir) if project_dir is not None else None\n\n    absolute_paths = [\n        str(resolve_path(path, base_dir=base_dir))\n        for path in df["image_path"].astype(str).tolist()\n    ]\n\n    if check_files:\n        missing_paths = [path for path in absolute_paths if not Path(path).exists()]\n        if missing_paths:\n            preview = "\\n".join(missing_paths[:10])\n            raise FileNotFoundError(\n                f"{len(missing_paths)} image files were not found. "\n                f"First missing paths:\\n{preview}\\n\\n"\n                "Check whether image_path is absolute or relative to PROJECT_DIR."\n            )\n\n    df["absolute_path"] = absolute_paths\n    return df\n\n\ndef load_dataframes(\n    config: PreprocessingConfig,\n) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, List[str]]:\n    """\n    Load train_folds.csv and test.csv, then create train/val/test DataFrames.\n\n    Returns:\n        train_df, val_df, test_df, class_names\n    """\n    split_dir = Path(config.split_dir)\n\n    train_folds_path = resolve_path(config.train_folds_csv, base_dir=split_dir)\n    test_path = resolve_path(config.test_csv, base_dir=split_dir)\n\n    train_folds_df = load_split_csv(\n        train_folds_path,\n        required_columns=TRAIN_FOLDS_REQUIRED_COLUMNS,\n        split_name="train_folds",\n    )\n    test_df = load_split_csv(\n        test_path,\n        required_columns=TEST_REQUIRED_COLUMNS,\n        split_name="test",\n    )\n\n    class_names = infer_class_names(train_folds_df, test_df)\n    train_df, val_df = split_train_val_from_folds(\n        train_folds_df=train_folds_df,\n        val_fold=config.val_fold,\n    )\n\n    train_df = add_absolute_paths(\n        train_df,\n        project_dir=config.project_dir,\n        check_files=config.check_files,\n    )\n    val_df = add_absolute_paths(\n        val_df,\n        project_dir=config.project_dir,\n        check_files=config.check_files,\n    )\n    test_df = add_absolute_paths(\n        test_df,\n        project_dir=config.project_dir,\n        check_files=config.check_files,\n    )\n\n    return train_df, val_df, test_df, class_names\n\n\ndef decode_image(image_path: tf.Tensor) -> tf.Tensor:\n    """Read an image file and decode it as RGB with 3 channels."""\n    image_bytes = tf.io.read_file(image_path)\n\n    image = tf.io.decode_image(\n        image_bytes,\n        channels=3,\n        expand_animations=False,\n    )\n    image.set_shape([None, None, 3])\n    return image\n\n\ndef preprocess_image(\n    image: tf.Tensor,\n    img_size: int = 224,\n    normalization: Optional[str] = None,\n) -> tf.Tensor:\n    """Resize image and cast to float32; normalization is optional."""\n    image = tf.image.resize(image, [img_size, img_size], method="bilinear")\n    image = tf.cast(image, tf.float32)\n\n    if normalization == "rescale_0_1":\n        image = image / 255.0\n    elif normalization is None:\n        pass\n    else:\n        raise ValueError(\n            "normalization must be None or \'rescale_0_1\'. "\n            f"Got: {normalization}"\n        )\n\n    image.set_shape([img_size, img_size, 3])\n    return image\n\n\ndef _load_and_preprocess_example(\n    image_path: tf.Tensor,\n    label: tf.Tensor,\n    img_size: int,\n    normalization: Optional[str],\n) -> Tuple[tf.Tensor, tf.Tensor]:\n    """Load one example and return image tensor plus label."""\n    image = decode_image(image_path)\n    image = preprocess_image(image, img_size=img_size, normalization=normalization)\n    label = tf.cast(label, tf.int32)\n    return image, label\n\n\ndef build_dataset_from_dataframe(\n    df: pd.DataFrame,\n    config: PreprocessingConfig,\n    training: bool = False,\n) -> tf.data.Dataset:\n    """Build a tf.data.Dataset from a DataFrame with file paths and labels."""\n    path_column = "absolute_path" if "absolute_path" in df.columns else "image_path"\n\n    image_paths = df[path_column].astype(str).tolist()\n    labels = df["label"].astype(int).tolist()\n\n    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))\n\n    if training and config.shuffle_train:\n        dataset = dataset.shuffle(\n            buffer_size=len(df),\n            seed=config.seed,\n            reshuffle_each_iteration=True,\n        )\n\n    dataset = dataset.map(\n        lambda image_path, label: _load_and_preprocess_example(\n            image_path=image_path,\n            label=label,\n            img_size=config.img_size,\n            normalization=config.normalization,\n        ),\n        num_parallel_calls=AUTOTUNE,\n    )\n\n    dataset = dataset.batch(config.batch_size)\n    dataset = dataset.prefetch(AUTOTUNE)\n\n    return dataset\n\n\ndef build_datasets(\n    config: PreprocessingConfig,\n) -> Tuple[tf.data.Dataset, tf.data.Dataset, tf.data.Dataset, List[str]]:\n    """\n    Build train_ds, val_ds and test_ds from CSV splits.\n\n    train_ds uses all rows where fold != config.val_fold.\n    val_ds uses all rows where fold == config.val_fold.\n    test_ds uses all rows from test.csv.\n    """\n    train_df, val_df, test_df, class_names = load_dataframes(config)\n\n    train_ds = build_dataset_from_dataframe(train_df, config=config, training=True)\n    val_ds = build_dataset_from_dataframe(val_df, config=config, training=False)\n    test_ds = build_dataset_from_dataframe(test_df, config=config, training=False)\n\n    return train_ds, val_ds, test_ds, class_names\n\n\ndef make_common_config(\n    project_dir: Union[str, Path] = "/content/drive/MyDrive/FDL Project",\n    val_fold: int = 0,\n    img_size: int = 224,\n    batch_size: int = 32,\n    normalization: Optional[str] = None,\n    shuffle_train: bool = True,\n    check_files: bool = True,\n    seed: int = 42,\n) -> PreprocessingConfig:\n    """\n    Create the standard project preprocessing configuration.\n\n    This wrapper fixes the common project split paths:\n    - data/splits/train_folds.csv\n    - data/splits/test.csv\n\n    Use this in modelling notebooks when you want the shared setup without\n    rewriting SPLIT_DIR, TRAIN_FOLDS_CSV, TEST_CSV and PreprocessingConfig.\n    """\n    project_dir = Path(project_dir)\n    split_dir = project_dir / "data" / "splits"\n\n    return PreprocessingConfig(\n        project_dir=str(project_dir),\n        split_dir=str(split_dir),\n        train_folds_csv="train_folds.csv",\n        test_csv="test.csv",\n        val_fold=val_fold,\n        img_size=img_size,\n        batch_size=batch_size,\n        seed=seed,\n        normalization=normalization,\n        shuffle_train=shuffle_train,\n        check_files=check_files,\n    )\n\n\ndef load_common_datasets(\n    project_dir: Union[str, Path] = "/content/drive/MyDrive/FDL Project",\n    val_fold: int = 0,\n    img_size: int = 224,\n    batch_size: int = 32,\n    normalization: Optional[str] = None,\n    shuffle_train: bool = True,\n    check_files: bool = True,\n    seed: int = 42,\n) -> Tuple[tf.data.Dataset, tf.data.Dataset, tf.data.Dataset, List[str]]:\n    """\n    Build the shared train, validation and test datasets with one function call.\n\n    Returns:\n        train_ds, val_ds, test_ds, class_names\n\n    Typical use in a modelling notebook:\n        train_ds, val_ds, test_ds, class_names = load_common_datasets(\n            project_dir="/content/drive/MyDrive/FDL Project",\n            val_fold=0,\n            img_size=224,\n            batch_size=32,\n            normalization=None,\n        )\n    """\n    config = make_common_config(\n        project_dir=project_dir,\n        val_fold=val_fold,\n        img_size=img_size,\n        batch_size=batch_size,\n        normalization=normalization,\n        shuffle_train=shuffle_train,\n        check_files=check_files,\n        seed=seed,\n    )\n    return build_datasets(config)\n\n\ndef summarize_dataset(\n    dataset: tf.data.Dataset,\n    class_names: Optional[List[str]] = None,\n    max_batches: int = 1,\n) -> None:\n    """Print basic information about one or more dataset batches."""\n    print("Dataset summary")\n    print("-" * 60)\n\n    if class_names is not None:\n        print(f"Number of classes: {len(class_names)}")\n        print(f"Class names: {class_names}")\n\n    for batch_idx, (images, labels) in enumerate(dataset.take(max_batches), start=1):\n        print(f"\\nBatch {batch_idx}")\n        print(f"images.shape: {images.shape}")\n        print(f"images.dtype: {images.dtype}")\n        print(f"labels.shape: {labels.shape}")\n        print(f"labels.dtype: {labels.dtype}")\n        print(f"image min: {tf.reduce_min(images).numpy():.3f}")\n        print(f"image max: {tf.reduce_max(images).numpy():.3f}")\n        print(f"first labels: {labels[:10].numpy()}")\n\n    print("-" * 60)\n\n\ndef summarize_split_dataframes(\n    train_df: pd.DataFrame,\n    val_df: pd.DataFrame,\n    test_df: pd.DataFrame,\n    class_names: Optional[List[str]] = None,\n) -> pd.DataFrame:\n    """Return a compact class distribution table for train, val and test."""\n    n_classes = len(class_names) if class_names is not None else int(\n        max(train_df["label"].max(), val_df["label"].max(), test_df["label"].max()) + 1\n    )\n\n    rows = []\n    for label in range(n_classes):\n        class_name = class_names[label] if class_names is not None else f"class_{label}"\n        rows.append(\n            {\n                "label": label,\n                "class_name": class_name,\n                "train": int((train_df["label"] == label).sum()),\n                "val": int((val_df["label"] == label).sum()),\n                "test": int((test_df["label"] == label).sum()),\n            }\n        )\n\n    summary = pd.DataFrame(rows)\n    total_row = pd.DataFrame(\n        [\n            {\n                "label": "total",\n                "class_name": "total",\n                "train": len(train_df),\n                "val": len(val_df),\n                "test": len(test_df),\n            }\n        ]\n    )\n    return pd.concat([summary, total_row], ignore_index=True)\n'

preprocessing_py_path.write_text(preprocessing_py_code, encoding="utf-8")

print(f"Saved: {preprocessing_py_path}")


Saved: /content/drive/MyDrive/FDL Project/scripts/preprocessing.py


## 14. Import dal modulo esportato e uso nei notebook di training

Dopo aver eseguito la cella di export, gli altri notebook possono importare `load_common_datasets` da `scripts/preprocessing.py` e creare direttamente `train_ds`, `val_ds`, `test_ds`, `class_names`.


In [ ]:
import sys
sys.path.append(f"{PROJECT_DIR}/scripts")

from preprocessing import (
    PreprocessingConfig,
    build_datasets,
    make_common_config,
    load_common_datasets,
)

print("Module import works.")

# Minimal code to use in a training notebook:
#
# train_ds, val_ds, test_ds, class_names = load_common_datasets(
#     project_dir=PROJECT_DIR,
#     val_fold=0,
#     img_size=224,
#     batch_size=32,
#     normalization=None,
# )
#
# Then:
# history = model.fit(train_ds, validation_data=val_ds, epochs=...)
# test_results = model.evaluate(test_ds)


Module import works.
